# Computational Pipeline: Acoustic Decoupling and Sociolinguistic Alignment
### Beyond Acoustic Imitation: Acoustic Decoupling and Sociolinguistic Alignment in Synthetic Vocal Performance
**Author:** Pegah Merrikhi (Independent Researcher)  
**Target Focus:** Speech Communication Revision / Sociolinguistics & Digital Humanities Package  

This Jupyter Notebook documents the complete analytical pipeline, from raw signal preprocessing and feature extraction to the statistical verification of terminal pitch dispersion, cross-feature acoustic decoupling, and dialectal co-occurrence dynamics (CCI).

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# Plot styling configuration
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

PALETTE = {
    'indigo': '#1A237E',
    'gold': '#FFB300',
    'pink': '#D81B60',
    'turquoise': '#00ACC1',
    'slate': '#455A64'
}

## 1. Raw Data Ingestion & Signal Feature Alignment
We load the raw frame-by-frame time series and raw continuous $F_0$ trajectory extracted via YIN (65–400 Hz range) at 22,050 Hz sampling rate.

In [ ]:
df_frames = pd.read_csv('/mnt/data/raw_frame_timeseries_full.csv')
df_f0 = pd.read_csv('/mnt/data/raw_f0_hz_full.csv')

print(f"Total frame observations: {len(df_frames)}")
print(f"Total F0 observations: {len(df_f0)}")
df_frames.head()

## 2. Temporal Segmentation (S1–S4) & Terminal F0 Dispersion
Reconstructing Table 2 to address the SPECOM-D-26-00622 reviewer critiques:
Explicitly computing variances across S1, S2, S3, and S4 to verify the terminal dispersion hypothesis.

In [ ]:
df_acoustic_summary = pd.read_csv('/mnt/data/acoustic_summary.csv')
display(df_acoustic_summary[['seg', 'start', 'end', 'nframes', 'f0_mean', 'f0_var', 'f0_var_he169', 'rms_mean', 'sc_mean', 'zcr_mean']])

# Compute terminal jump ratio
s1_var = df_acoustic_summary.loc[df_acoustic_summary['seg'] == 'S1', 'f0_var'].values[0]
s4_var = df_acoustic_summary.loc[df_acoustic_summary['seg'] == 'S4', 'f0_var'].values[0]
s1_3_mean_var = df_acoustic_summary.loc[df_acoustic_summary['seg'].isin(['S1', 'S2', 'S3']), 'f0_var'].mean()

print(f"S4 / S(1-3) Variance Ratio: {s4_var / s1_3_mean_var:.2f}x")
print(f"S4 / S1 Variance Ratio: {s4_var / s1_var:.2f}x")

## 3. Statistical Testing: Levene's Test & Fisher's z Transformation
Evaluating variance homogeneity across temporal segments and measuring acoustic decoupling.

In [ ]:
with open('/mnt/data/comparative_stats_summary.json') as f:
    stats_data = json.load(f)

print("Inferential Statistics Summary:")
for k, v in stats_data.items():
    print(f"Key: {k}")
    if isinstance(v, dict):
        for sub_k, sub_v in v.items():
            print(f"  - {sub_k}: {sub_v}")
    else:
        print(f"  - {v}")

## 4. Sociolinguistic Alignment: Code-Co-occurrence Index (CCI)
Evaluating convergence between African American Vernacular English (AAVE) and Nigerian Pidgin (NigP) across 16 analytical windows.

In [ ]:
df_cci = pd.read_csv('/mnt/data/code_counts_demo.csv', comment='#')
# Compute CCI if not explicit: 2 * min(code_a, code_b) / (code_a + code_b)
df_cci['cci'] = 2 * df_cci[['code_a', 'code_b']].min(axis=1) / (df_cci['code_a'] + df_cci['code_b'])
display(df_cci)

# Visualization of CCI trajectory
plt.figure(figsize=(9, 4.5))
plt.plot(df_cci['window_id'], df_cci['cci'], marker='o', color=PALETTE['turquoise'], lw=2.5, label='CCI Trajectory')
plt.axhline(1.0, color=PALETTE['pink'], linestyle='--', label='Equi-distribution Target (1.00)')
plt.title("Sociolinguistic Dialectal Co-occurrence Convergence (AAVE - NigP)", fontsize=12, fontweight='bold', color=PALETTE['indigo'])
plt.xlabel("Temporal Analysis Window (W01 - W16)")
plt.ylabel("Co-occurrence Index (CCI)")
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()